In [2]:
import socket

ip = "8.8.8.8"
port = 53  # DNS

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(5)

try:
    sock.sendto(b"test", (ip, port))
    print("UDP packet sent")
except Exception as e:
    print(f"Error: {e}")

sock.close()

UDP packet sent


In [ ]:
import socket

server = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
server.bind(("0.0.0.0", 9999))

while True:
    data, addr = server.recvfrom(1024)
    print(f"Received: {data} from {addr}")
    server.sendto(b"ACK", addr)

In [1]:
import socket

client = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
client.settimeout(3)

client.sendto(b"hello", ("127.0.0.1", 9999))

try:
    data, addr = client.recvfrom(1024)
    print(data.decode())
except socket.timeout:
    print("No response")

ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

In [2]:
import socket

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(3)

try:
    sock.sendto(b"\x00", ("8.8.8.8", 53))
    print("DNS UDP port reachable")
except Exception as e:
    print(e)

sock.close()

DNS UDP port reachable


In [3]:
import socket

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(3)

try:
    sock.sendto(b'\x1b' + 47 * b'\0', ('time.google.com', 123))
    data, _ = sock.recvfrom(1024)

    print("NTP server responded")
except socket.timeout:
    print("NTP timeout")

NTP server responded


Since you work in WAN/Lab automation, UDP testing is very useful because ping only tests ICMP and TCP tests only TCP connectivity. Many real applications use UDP (DNS, NTP, Syslog, SNMP, RTP, QUIC, etc.).

1. Test UDP Port Reachability with Python

Unlike TCP, UDP doesn't have a handshake, so success can be harder to determine.

Example: Send UDP Packet
import socket

ip = "8.8.8.8"
port = 53  # DNS

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(5)

try:
    sock.sendto(b"test", (ip, port))
    print("UDP packet sent")
except Exception as e:
    print(f"Error: {e}")

sock.close()

2. UDP Echo Server Test
Server
import socket

server = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
server.bind(("0.0.0.0", 9999))

while True:
    data, addr = server.recvfrom(1024)
    print(f"Received: {data} from {addr}")
    server.sendto(b"ACK", addr)

Client
import socket

client = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
client.settimeout(3)

client.sendto(b"hello", ("127.0.0.1", 9999))

try:
    data, addr = client.recvfrom(1024)
    print(data.decode())
except socket.timeout:
    print("No response")

3. Test DNS UDP Service (Real World)

Your example IP 8.8.8.8 is Google's DNS server.

import socket

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(3)

try:
    sock.sendto(b"\x00", ("8.8.8.8", 53))
    print("DNS UDP port reachable")
except Exception as e:
    print(e)

sock.close()

4. NTP UDP Test (Port 123)

Common in network validation.

import socket

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(3)

try:
    sock.sendto(b'\x1b' + 47 * b'\0', ('time.google.com', 123))
    data, _ = sock.recvfrom(1024)

    print("NTP server responded")
except socket.timeout:
    print("NTP timeout")

5. Using netcat (Windows/Linux)

UDP listener:

nc -u -l 9999


UDP sender:

echo "hello" | nc -u 192.168.1.10 9999

6. Using iperf3 for UDP Testing

This is the most common WAN/Lab testing tool.

Start Server
iperf3 -s

UDP Client
iperf3 -c 10.1.1.1 -u

UDP at 100 Mbps
iperf3 -c 10.1.1.1 -u -b 100M

UDP for 30 Seconds
iperf3 -c 10.1.1.1 -u -b 100M -t 30


Output includes:

Jitter
Packet Loss
Bandwidth
Transfer Rate


Very useful for validating:

MPLS paths
WAN links
QoS policies
Overload/Unoverload testing
Traffic convergence tests
7. Parse UDP Test Results

Example:

result = {
    "destination": "8.8.8.8",
    "udp_port": 53,
    "reachable": True,
    "response_time_ms": 3
}

print(result)


Output:

{
    'destination': '8.8.8.8',
    'udp_port': 53,
    'reachable': True,
    'response_time_ms': 3
}

Common UDP Ports You'll See in Networking
Protocol	PortDNS	53
NTP	123
SNMP	161
Syslog	514
DHCP	67/68
VXLAN	4789
QUIC	443/UDP
BFD	3784/3785

For a network automation engineer, the most practical UDP tests to learn are:

DNS (UDP 53)
NTP (UDP 123)
Syslog (UDP 514)
SNMP (UDP 161)
iperf3 UDP throughput testing for WAN validation.

In [4]:
!pip install dnspython


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://packagefeedproxy.microsoft.io/pypi/simple/


In [5]:
import dns.message
import dns.query

dns_server = "8.8.8.8"

query = dns.message.make_query("google.com", "A")

try:
    response = dns.query.udp(
        query,
        dns_server,
        timeout=3
    )

    print("UDP DNS test successful")
    print(response)

except Exception as e:
    print("UDP DNS test failed")
    print(e)

UDP DNS test successful
id 59105
opcode QUERY
rcode NOERROR
flags QR RD RA
;QUESTION
google.com. IN A
;ANSWER
google.com. 300 IN A 142.251.33.206
;AUTHORITY
;ADDITIONAL


In [6]:
import socket

server_ip = "192.168.1.100"
server_port = 5000

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(3)

message = b"Hello from UDP client"

try:
    sock.sendto(message, (server_ip, server_port))

    data, address = sock.recvfrom(1024)

    print("Response:", data.decode())
    print("Received from:", address)

except socket.timeout:
    print("No UDP response received")

finally:
    sock.close()

No UDP response received


In [ ]:
import socket

server_ip = "0.0.0.0"
server_port = 5000

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

sock.bind((server_ip, server_port))

print(f"Listening on UDP {server_port}...")

while True:
    data, address = sock.recvfrom(1024)

    print(f"Received from {address}: {data.decode()}")

    sock.sendto(b"UDP packet received!", address)

Listening on UDP 5000...


In [1]:
import socket
import time

server_ip = "0.0.0.0"
server_port = 5000

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.bind((server_ip, server_port))

# Prevent recvfrom() from blocking forever
sock.settimeout(1)

print(f"Listening on UDP {server_port}...")

start_time = time.time()

while True:
    # Stop after 60 seconds
    if time.time() - start_time >= 60:
        print("Server timeout reached (60 seconds). Exiting...")
        break

    try:
        data, address = sock.recvfrom(1024)

        print(f"Received from {address}: {data.decode()}")

        sock.sendto(b"UDP packet received!", address)

        # Wait 10 seconds before processing next packet
        print("Sleeping for 10 seconds...")
        time.sleep(10)

    except socket.timeout:
        # No packet received within 1 second
        continue

sock.close()
print("UDP server stopped.")

Listening on UDP 5000...
Server timeout reached (60 seconds). Exiting...
UDP server stopped.


In [2]:
import socket

server = ("192.168.1.100", 5000)

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(5)

sock.sendto(b"Network Engineer Test", server)

try:
    data, addr = sock.recvfrom(1024)
    print("Server response:", data.decode())

except socket.timeout:
    print("No response")

sock.close()

No response


Yes. UDP testing is a little different from TCP because UDP is **connectionless**. There is no TCP-style three-way handshake, so a UDP test often requires the destination application to send a response before you can confidently say the UDP service is reachable.

For a network engineer, these are useful UDP testing methods.

### 1. Test DNS over UDP with PowerShell

DNS normally uses UDP port **53** for standard queries.

```powershell
Resolve-DnsName google.com -Server 8.8.8.8
```

You can also use:

```powershell
nslookup google.com 8.8.8.8
```

This is a practical UDP test because you send a DNS request and expect a valid DNS response.

You might see:

```text
Name:    google.com
Addresses:  2607:f8b0:...
            142.250.x.x
```

That tells you much more than simply sending an arbitrary UDP packet to port 53.

### 2. Python UDP DNS test

Python's `dnspython` library is useful for this:

```powershell
pip install dnspython
```

Then:

```python
import dns.message
import dns.query

dns_server = "8.8.8.8"

query = dns.message.make_query("google.com", "A")

try:
    response = dns.query.udp(
        query,
        dns_server,
        timeout=3
    )

    print("UDP DNS test successful")
    print(response)

except Exception as e:
    print("UDP DNS test failed")
    print(e)
```

The flow is:

```text
Python
   |
   | UDP/53 DNS Query
   v
8.8.8.8
   |
   | UDP/53 DNS Response
   v
Python
```

### 3. Basic UDP client using Python sockets

For generic UDP testing:

```python
import socket

server_ip = "192.168.1.100"
server_port = 5000

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(3)

message = b"Hello from UDP client"

try:
    sock.sendto(message, (server_ip, server_port))

    data, address = sock.recvfrom(1024)

    print("Response:", data.decode())
    print("Received from:", address)

except socket.timeout:
    print("No UDP response received")

finally:
    sock.close()
```

Notice:

```python
socket.SOCK_DGRAM
```

means **UDP**.

For TCP you would normally use:

```python
socket.SOCK_STREAM
```

### 4. Create your own UDP server for testing

This is excellent for learning because you control both ends.

On **PC/server 1**:

```python
import socket

server_ip = "0.0.0.0"
server_port = 5000

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

sock.bind((server_ip, server_port))

print(f"Listening on UDP {server_port}...")

while True:
    data, address = sock.recvfrom(1024)

    print(f"Received from {address}: {data.decode()}")

    sock.sendto(b"UDP packet received!", address)
```

Then on **PC/client 2**:

```python
import socket

server = ("192.168.1.100", 5000)

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.settimeout(5)

sock.sendto(b"Network Engineer Test", server)

try:
    data, addr = sock.recvfrom(1024)
    print("Server response:", data.decode())

except socket.timeout:
    print("No response")

sock.close()
```

Now you're actually testing:

```text
Client                            Server
192.168.1.10                     192.168.1.100
     |                                  |
     | ------ UDP dst port 5000 ------> |
     |                                  |
     | <----- UDP response ------------ |
     |                                  |
```

### 5. Use `iperf3` for UDP network testing

For actual network troubleshooting, `iperf3` is extremely useful.

On the server:

```powershell
iperf3 -s
```

On the client, TCP is the default:

```powershell
iperf3 -c 192.168.1.100
```

To specifically test **UDP**:

```powershell
iperf3 -c 192.168.1.100 -u
```

Set UDP bandwidth:

```powershell
iperf3 -c 192.168.1.100 -u -b 10M
```

Test 100 Mbps UDP:

```powershell
iperf3 -c 192.168.1.100 -u -b 100M
```

Run for 30 seconds:

```powershell
iperf3 -c 192.168.1.100 -u -b 100M -t 30
```

This gives you useful networking information such as:

```text
Bandwidth
Packet loss
Lost/total datagrams
Jitter
Throughput
```

This is much closer to what you'd use when investigating real network performance.

### 6. Why `Test-NetConnection -Port` isn't a UDP test

This is important.

Your earlier command:

```powershell
tnc 8.8.8.8 -Port 443
```

tests **TCP/443**.

You cannot simply do something like:

```powershell
tnc 8.8.8.8 -Port 53 -UDP
```

because `Test-NetConnection` doesn't provide a generic `-UDP` equivalent to its TCP `-Port` test.

So choose the tool based on what you're testing:

| Goal                      | Command/tool                                 |
| ------------------------- | -------------------------------------------- |
| ICMP reachability         | `ping 8.8.8.8`                               |
| TCP/443                   | `Test-NetConnection 8.8.8.8 -Port 443`       |
| UDP DNS/53                | `nslookup google.com 8.8.8.8`                |
| UDP DNS/53                | `Resolve-DnsName google.com -Server 8.8.8.8` |
| Generic UDP               | Python `socket.SOCK_DGRAM`                   |
| UDP performance           | `iperf3 -u`                                  |
| Packet-level verification | Wireshark                                    |

For your Python/network-automation learning, a good next exercise is to build one Python script that accepts an IP and performs **ICMP ping + TCP port test + UDP DNS test**, then prints a small PASS/FAIL connectivity report.


In [3]:
import subprocess
import socket

# -----------------------------
# ICMP TEST
# -----------------------------
def ping_test(ip):
    print(f"\n[ICMP] Testing ping to {ip}...")

    result = subprocess.run(
        ["ping", "-n", "2", ip],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print(f"PASS: {ip} is reachable using ICMP")
        return True
    else:
        print(f"FAIL: {ip} is not reachable using ICMP")
        return False


# -----------------------------
# TCP TEST
# -----------------------------
def tcp_test(ip, port):
    print(f"\n[TCP] Testing {ip}:{port}...")

    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(3)

        result = sock.connect_ex((ip, port))

        sock.close()

        if result == 0:
            print(f"PASS: TCP {ip}:{port} is reachable")
            return True
        else:
            print(f"FAIL: TCP {ip}:{port} is not reachable")
            return False

    except Exception as error:
        print(f"ERROR: {error}")
        return False


# -----------------------------
# UDP DNS TEST
# -----------------------------
def udp_dns_test(dns_server):
    print(f"\n[UDP] Testing DNS against {dns_server}:53...")

    # Raw DNS query for google.com A record
    dns_query = (
        b"\x12\x34"          # Transaction ID
        b"\x01\x00"          # Standard query
        b"\x00\x01"          # Questions = 1
        b"\x00\x00"          # Answer RRs
        b"\x00\x00"          # Authority RRs
        b"\x00\x00"          # Additional RRs
        b"\x06google"
        b"\x03com"
        b"\x00"
        b"\x00\x01"          # Type A
        b"\x00\x01"          # Class IN
    )

    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.settimeout(3)

    try:
        sock.sendto(
            dns_query,
            (dns_server, 53)
        )

        data, address = sock.recvfrom(4096)

        print(
            f"PASS: UDP DNS response received "
            f"from {address[0]}:{address[1]}"
        )

        print(f"Response size: {len(data)} bytes")

        return True

    except socket.timeout:
        print("FAIL: UDP DNS request timed out")
        return False

    except Exception as error:
        print(f"ERROR: {error}")
        return False

    finally:
        sock.close()


# -----------------------------
# MAIN PROGRAM
# -----------------------------

target_ip = "8.8.8.8"

icmp_result = ping_test(target_ip)

tcp_result = tcp_test(
    target_ip,
    443
)

udp_result = udp_dns_test(
    target_ip
)


# -----------------------------
# SUMMARY
# -----------------------------

print("\n" + "=" * 40)
print("NETWORK CONNECTIVITY REPORT")
print("=" * 40)

print(
    f"ICMP Ping        : "
    f"{'PASS' if icmp_result else 'FAIL'}"
)

print(
    f"TCP 443          : "
    f"{'PASS' if tcp_result else 'FAIL'}"
)

print(
    f"UDP DNS Port 53  : "
    f"{'PASS' if udp_result else 'FAIL'}"
)


[ICMP] Testing ping to 8.8.8.8...
FAIL: 8.8.8.8 is not reachable using ICMP

[TCP] Testing 8.8.8.8:443...
PASS: TCP 8.8.8.8:443 is reachable

[UDP] Testing DNS against 8.8.8.8:53...
PASS: UDP DNS response received from 8.8.8.8:53
Response size: 44 bytes

NETWORK CONNECTIVITY REPORT
ICMP Ping        : FAIL
TCP 443          : PASS
UDP DNS Port 53  : PASS
